In [ ]:
# !pip install datasets
!pip install transformers
!pip install torch

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer


In [3]:
train_path = "training_data/NLI/train.csv"
dev_path = "training_data/NLI/dev.csv"

In [4]:
train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

In [5]:
print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print(train_df.columns)
print(train_df.head(5))
print(train_df["label"].unique())
print(train_df["label"].value_counts())
print(train_df.isnull().sum())
print(train_df["premise"].str.len().describe())
print(train_df["hypothesis"].str.len().describe())
print(train_df.iloc[0])

Train shape: (24432, 3)
Dev shape: (6736, 3)
Index(['premise', 'hypothesis', 'label'], dtype='str')
                                             premise  \
0  yeah i don't know cut California in half or so...   
1                      actual names will not be used   
2          The film was directed by Randall Wallace.   
3   "How d'you know he'll sign me on?"Anse studie...   
4  In the light of the candles his cheeks looked ...   

                                          hypothesis  label  
0  Yeah. I'm not sure how to make that fit. Maybe...      1  
1  For the sake of privacy, actual names are not ...      1  
2  The film was directed by Randall Wallace and s...      1  
3       Anse looked at himself in a cracked mirror.       1  
4  Drew regarded his best friend and noted that i...      1  
[1 0]
label
1    12648
0    11784
Name: count, dtype: int64
premise       0
hypothesis    0
label         0
dtype: int64
count    24432.000000
mean       109.350851
std         74.169780
min 

In [6]:
train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)

print(train_dataset)

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 24432
})


In [7]:

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [8]:
def tokenize(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [9]:
train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

In [10]:
import torch
print(torch.__version__)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

2.10.0+cu128


In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="logs",
    report_to="none",
    disable_tqdm=False,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_f1 = f1_score(labels, preds, average="macro")

    return {"macro_f1": macro_f1}

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()
# trainer.evaluate()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.437182,0.424282,0.807628
2,0.286678,0.444143,0.825109
3,0.155759,0.659946,0.826568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4581, training_loss=0.32232934318497514, metrics={'train_runtime': 300.7446, 'train_samples_per_second': 243.715, 'train_steps_per_second': 15.232, 'total_flos': 4821246978416640.0, 'train_loss': 0.32232934318497514, 'epoch': 3.0})